# Day 10: アライメントと定量

## この章で解く現実の課題

短いreadを、どの遺伝子から来たRNA断片として数えるのかを理解します。

count matrixは最初から存在する表ではありません。FASTQ readを参照ゲノムまたはトランスクリプトームに対応づけ、遺伝子ごとに集計して作られます。


## キーワード辞典（この章）

| キーワード | まずこう理解する | この章での使いどころ | よくある誤解 |
|---|---|---|---|
| alignment | readを参照配列へ対応づける処理 | read起源推定の基礎 | mapできたreadは全て正しいと考える |
| reference genome | 対応づけ先の標準配列 | マッピング基準 | 参照版の違いを無視する |
| annotation (GTF) | 遺伝子領域定義ファイル | gene-level集計に必要 | 参照配列だけで十分と思う |
| mapping rate | map成功readの割合 | データ品質・参照適合性の指標 | 高ければ必ず良解析と思う |
| quantification | 遺伝子/転写産物ごとの量推定 | count matrix生成 | alignmentと同義で使う |
| unmapped read | 参照へ対応できないread | QC/汚染確認対象 | 全てノイズとして無視すればよいと思う |



## この章での判断軸

1. FASTQ→mapping→countの連鎖で説明できるか確認する。
2. mapping rateだけでなく、どのreadが落ちたかも見る。
3. 参照配列とannotationの整合を保つ。


## 2つの代表的な考え方

- alignment based: readをゲノムに対応づけ、その位置がどの遺伝子に重なるか数える。例: STAR + featureCounts。
- transcriptome quantification: readがどの転写産物に由来するかを高速に推定する。例: Salmon, Kallisto。

どちらも最終的には「遺伝子または転写産物ごとの量」を作るための処理です。


In [ ]:
import pandas as pd

reads = pd.DataFrame({
    "read_id": ["r1", "r2", "r3", "r4", "r5", "r6", "r7"],
    "sequence": ["ACGT", "CGTT", "TTAA", "GGCA", "ACGT", "CCCC", "TTAA"],
})

reference_hits = pd.DataFrame({
    "read_id": ["r1", "r2", "r3", "r4", "r5", "r6", "r7"],
    "mapped_gene": ["IL6", "IL6", "TNF", "ACTB", "IL6", None, "TNF"],
    "mapping_status": ["unique", "unique", "unique", "unique", "unique", "unmapped", "unique"],
})

reads.merge(reference_hits, on="read_id")


In [ ]:
# 遺伝子ごとにmapped readを数える。
# 実際のfeatureCountsも、概念的にはreadをfeatureごとに集計している。
count_matrix_one_sample = (
    reference_hits.dropna(subset=["mapped_gene"])
    .groupby("mapped_gene")
    .size()
    .rename("sample_1_count")
    .to_frame()
)
count_matrix_one_sample


In [ ]:
mapping_rate = (reference_hits["mapping_status"] != "unmapped").mean()
print(f"mapping rate: {mapping_rate:.1%}")


## 読み取り

`r6` はunmappedなので、どの遺伝子にも数えません。`r1`, `r2`, `r5` は `IL6` に対応づいたので、`IL6` のcountは3になります。

実際のRNA-seqではreadは数千万本あり、参照ゲノム、遺伝子アノテーション、マッピング品質、multi-mapping readなどを考慮します。ここでは、count matrixが「readを遺伝子に対応づけて集計した表」であることを押さえるのが目的です。


In [ ]:
pipeline = pd.DataFrame({
    "step": ["FASTQ", "QC", "alignment or quantification", "gene-level count", "count matrix"],
    "example_tool": ["sequencer output", "FastQC / MultiQC", "STAR / Salmon", "featureCounts / tximport", "pandas / DESeq2 input"],
    "main_question": [
        "どんなreadが得られたか",
        "read品質は十分か",
        "readはどこ由来か",
        "遺伝子ごとに何readあるか",
        "サンプル間で比較できる表か",
    ],
})
pipeline


## エラーが出た場合

- `None` が表に残る: unmapped readを除外するには `dropna(subset=["mapped_gene"])` を使います。
- countが期待と違う: groupby前の表でreadがどのgeneに対応しているか確認します。

## この章のゴール

FASTQからcount matrixまでの流れを、read、参照配列、遺伝子ごとの集計という言葉で説明できれば合格です。
